# EthicCheck AI Topic 4 data quality audit

Run **Runtime → Run all** in Google Colab. This notebook downloads the repository's `evaluation/audit.py` file, checks the four input files against the GitHub snapshot cited in the report, and runs both validation rules on the eight prepared records. It uses only the Python standard library.

Screenshot the first output for **Figure 1 (Validation output)**. Screenshot the second output for **Figure 2 (Supporting evidence)**. Open `audit.py` in the Colab **Files** panel to view the downloaded script.

In [ ]:
import json
from pathlib import Path
import runpy
from urllib.request import urlopen

project_url = "https://raw.githubusercontent.com/nafizshuk/ai-engineering-ethiccheck-ai"
script = Path("audit.py")
with urlopen(f"{project_url}/main/evaluation/audit.py", timeout=30) as response:
    script.write_bytes(response.read())

module = runpy.run_path(str(script))
inputs = module["load_inputs"](None)
results = module["audit"](inputs)
results_file = Path("topic4_audit_results.json")
results_file.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
module["print_results"](results)

## Supporting evidence

The privacy example shows the original TXT source, its line reference, the cited NIST section, and the corresponding prepared text.

In [ ]:
from textwrap import fill

record = results["supporting_record"]
source = inputs[record["source"]].decode("utf-8-sig").splitlines()
lo, hi = record["line_start"], record["line_end"]

print("EthicCheck AI | Supporting evidence")
print(f"Prepared record: {record['record_id']}")
print(f"Source: {record['source']}, {'line '+str(lo) if lo == hi else f'lines {lo}-{hi}'}")
print(f"Reference: {record['reference_title']} §{record['reference_section']}")
print()
print("Original source text")
print(fill(" ".join(source[lo-1:hi]), width=92))
print()
print("Prepared record text")
print(fill(record["text"].strip(), width=92))
print()
print("Source and prepared text match after normalisation:",
      module["normalise"](" ".join(source[lo-1:hi])) == module["normalise"](record["text"]))